# Milestone 4 — Initial Insights | Davao Commute Router

This notebook audits **OpenStreetMap route and stop mapping completeness**, not the speed or fares of real public transport. It uses only the committed Milestone 3 CSVs and makes no network requests.

**Learning goal:** Reproduce the four figures below and explain why they do *not* establish current operational routes or valid passenger transfers. Data: © OpenStreetMap contributors (ODbL).


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'm4_insights.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'm4_insights.py').exists(), 'Run this notebook from the repository or notebooks/ directory'
sys.path.insert(0, str(ROOT))
from scripts.m4_insights import (load_data, summarise, plot_route_coverage,
                                 plot_metadata_quality, plot_coordinate_spread,
                                 plot_shared_stops)
stops, routes, route_stops = load_data(ROOT / 'data' / 'processed')
from io import BytesIO
from IPython.display import Image, display

def show_chart(fig):
    data = BytesIO()
    fig.savefig(data, format='png', dpi=140, facecolor='white')
    display(Image(data=data.getvalue()))
    plt.close(fig)

summary = summarise(stops, routes, route_stops)
summary


## Figure 1 — Ordered stop coverage by route

**Question:** Which OSM route relations have explicit stop sequences? **Interpretation:** A missing stop sequence measures a data-mapping gap; it does not establish that the vehicle does not stop or that the route is inactive. Some mapped routes are intercity and ferry services.


In [ ]:
fig = plot_route_coverage(routes, route_stops)
show_chart(fig)


## Figure 2 — Stop metadata quality

**Question:** How many mapped stop records contain usable names and point coordinates? **Interpretation:** This indicates route-search readiness, not whether real stops physically exist.


In [ ]:
fig = plot_metadata_quality(stops)
show_chart(fig)


## Figure 3 — Geographic coverage warning

**Question:** Do all point coordinates appear local to Davao City? **Interpretation:** No. Route relation members may be returned outside the geographic filter; this scatterplot is not a road map or administrative boundary.


In [ ]:
fig = plot_coordinate_spread(stops)
show_chart(fig)


## Figure 4 — Shared route-member stop IDs

**Question:** Which stop IDs occur in more than one OSM route relation? **Interpretation:** They are potential transfer references, **not verified transfers**. Check locations, schedules, directions, city scope, and legal service rules before using them in routing.


In [ ]:
fig = plot_shared_stops(stops, route_stops)
show_chart(fig)


## Cautious inference

The existing data is more useful for measuring **data availability and data quality** than for estimating door-to-door journeys. It contains no verified fares, schedules, walking-network edges, tricycle zones, or traffic profiles. Claims about the city's real-world service coverage, best route, waiting time or lowest fare would be unsupported.

Next data priorities: local-only route geometry and ordered stops; authoritative operating status; regulated fares and schedules; pedestrian network and verified service-zone rules. For detailed quantitative interpretations, see [`docs/MILESTONE4_INSIGHTS.md`](../docs/MILESTONE4_INSIGHTS.md).
